# CrashSignal -- Day 1: Data Pipeline & Live Stress Detection

**DigitalOcean Gradient AI Hackathon**

A live market stress detection system that combines:
1. 30 years of historical stress indicators (training data)
2. Live market data updated daily (inference input)
3. Live financial news sentiment (additional signal)

The system answers one question in real time:
> *How stressed is the market RIGHT NOW compared to every crisis since 1990?*

---

**Day 1 Goals:**
- Pull 30 years of all 15 stress indicators from FRED
- Pull live market data from Yahoo Finance
- Pull and analyze today's financial news sentiment
- Clean and align everything to trading days
- Label all historical crisis periods
- Visualize historical stress vs crises
- Save clean dataset ready for Day 2 training

---
## Step 1 -- Environment Setup

Install all required packages and configure global settings.
We use `fredapi` to pull macroeconomic data from the Federal Reserve,
`yfinance` for live market prices, `transformers` for FinBERT sentiment,
and `plotly` for interactive dark-themed visualizations.

In [ ]:
!pip install -q fredapi yfinance
!pip install -q pandas-datareader requests beautifulsoup4
!pip install -q transformers torch
!pip install -q matplotlib seaborn plotly
!pip install -q scikit-learn numpy pandas tqdm

In [ ]:
import os
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
from fredapi import Fred
from datetime import datetime, timedelta
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

plt.style.use("dark_background")

START_DATE = "1994-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")
TODAY      = END_DATE

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Date range: {START_DATE} to {TODAY}")
print(f"This includes TODAY's live data.")

---
## Step 2 -- FRED API Setup

The Federal Reserve Economic Data (FRED) API gives us free access to
thousands of economic time series. We use it for 11 of our 15 stress
indicators. Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
FRED_API_KEY = "idk"  # Replace with your FRED API key
fred = Fred(api_key=FRED_API_KEY)

# Quick connectivity test
vix_test = fred.get_series("VIXCLS", observation_start="2024-01-01")
print(f"FRED connected.")
print(f"Latest VIX date: {vix_test.index[-1].date()}")
print(f"Latest VIX value: {vix_test.iloc[-1]:.2f}")

---
## Step 3 -- Pull All 15 Stress Indicators

We pull 15 market stress indicators spanning volatility, credit markets,
yield curves, financial conditions, employment, consumer sentiment, and
safe-haven flows. Each indicator has a defined stress direction (whether
high or low values signal stress) and a threshold level.

In [ ]:
indicators_config = {
    "vix": {
        "source": "fred", "series": "VIXCLS",
        "name": "VIX Volatility Index",
        "description": "Market fear gauge. >30 = stress.",
        "stress_direction": "high", "stress_threshold": 30
    },
    "ted_spread": {
        "source": "fred", "series": "TEDRATE",
        "name": "TED Spread",
        "description": "Interbank lending stress. >1% = concern.",
        "stress_direction": "high", "stress_threshold": 1.0
    },
    "yield_curve": {
        "source": "fred", "series": "T10Y2Y",
        "name": "10Y-2Y Yield Curve",
        "description": "Negative = recession signal.",
        "stress_direction": "low", "stress_threshold": 0.0
    },
    "credit_spread_hy": {
        "source": "fred", "series": "BAMLH0A0HYM2",
        "name": "High Yield Credit Spread",
        "description": "Junk bond stress. >8% = crisis.",
        "stress_direction": "high", "stress_threshold": 8.0
    },
    "credit_spread_ig": {
        "source": "fred", "series": "BAMLC0A0CM",
        "name": "Investment Grade Credit Spread",
        "description": "Corporate bond stress.",
        "stress_direction": "high", "stress_threshold": 2.0
    },
    "financial_conditions": {
        "source": "fred", "series": "NFCI",
        "name": "National Financial Conditions Index",
        "description": "Composite stress. Positive = tighter.",
        "stress_direction": "high", "stress_threshold": 0.0
    },
    "bank_stress": {
        "source": "fred", "series": "DPCREDIT",
        "name": "Discount Window Borrowing",
        "description": "Banks borrowing from Fed = stress.",
        "stress_direction": "high", "stress_threshold": 1000
    },
    "unemployment_claims": {
        "source": "fred", "series": "ICSA",
        "name": "Initial Jobless Claims",
        "description": "Spike = economic deterioration.",
        "stress_direction": "high", "stress_threshold": 400000
    },
    "consumer_sentiment": {
        "source": "fred", "series": "UMCSENT",
        "name": "Consumer Sentiment",
        "description": "Low = recession fear.",
        "stress_direction": "low", "stress_threshold": 60
    },
    "manufacturing_pmi": {
        "source": "fred", "series": "MANEMP",
        "name": "Manufacturing Employment",
        "description": "Declining = contraction.",
        "stress_direction": "low", "stress_threshold": 12000
    },
    "real_rates": {
        "source": "fred", "series": "DFII10",
        "name": "10Y Real Interest Rate (TIPS)",
        "description": "Very negative = distorted markets.",
        "stress_direction": "low", "stress_threshold": -1.0
    },
    "dollar_index": {
        "source": "yahoo", "ticker": "DX-Y.NYB",
        "name": "US Dollar Index (DXY)",
        "description": "Spike = flight to safety.",
        "stress_direction": "high", "stress_threshold": 105
    },
    "gold_ratio": {
        "source": "yahoo", "ticker": "GLD",
        "name": "Gold ETF Price",
        "description": "Rising relative to equities = fear.",
        "stress_direction": "high", "stress_threshold": 180
    },
    "vvix": {
        "source": "yahoo", "ticker": "^VVIX",
        "name": "VIX of VIX (VVIX)",
        "description": "Volatility of volatility. >120 = extreme.",
        "stress_direction": "high", "stress_threshold": 120
    },
    "sp500": {
        "source": "yahoo", "ticker": "^GSPC",
        "name": "S&P 500",
        "description": "Used for drawdown calculation.",
        "stress_direction": "low", "stress_threshold": None
    }
}

In [ ]:
def pull_all_indicators(config, start, end):
    """
    Pull all 15 indicators from FRED and Yahoo Finance.
    Returns dict of {name: pd.Series}.
    Each indicator is fetched with full history from start to end.
    Errors are caught gracefully so one failure does not block the rest.
    """
    data = {}

    for key, cfg in tqdm(config.items(), desc="Pulling indicators"):
        try:
            if cfg["source"] == "fred":
                series = fred.get_series(
                    cfg["series"],
                    observation_start=start,
                    observation_end=end
                )
                series = series.dropna()
                data[key] = series
                print(
                    f"  [OK] {cfg['name']:40s} "
                    f"rows={len(series):5d}  "
                    f"latest={series.iloc[-1]:.2f}  "
                    f"({series.index[-1].date()})"
                )

            elif cfg["source"] == "yahoo":
                ticker = yf.Ticker(cfg["ticker"])
                hist = ticker.history(start=start, end=end)["Close"]
                hist = hist.dropna()
                data[key] = hist
                print(
                    f"  [OK] {cfg['name']:40s} "
                    f"rows={len(hist):5d}  "
                    f"latest={hist.iloc[-1]:.2f}  "
                    f"({hist.index[-1].date()})"
                )

        except Exception as e:
            print(f"  [FAIL] {key}: {e}")
            data[key] = pd.Series(dtype=float)

    return data


raw_data = pull_all_indicators(indicators_config, START_DATE, TODAY)

# Summary
success = sum(1 for v in raw_data.values() if len(v) > 0)
failed  = sum(1 for v in raw_data.values() if len(v) == 0)
print(f"\n{'='*60}")
print(f"Total indicators pulled: {success}/{len(raw_data)}")
if failed > 0:
    print(f"Failed: {[k for k, v in raw_data.items() if len(v) == 0]}")
print(f"{'='*60}")

---
## Step 4 -- Pull Live Data for Today

This cell powers the live dashboard. It pulls the most recent value of
every market indicator using Yahoo Finance and FRED. At inference time in
production, this function runs once per day to feed the trained model.

In [ ]:
def get_live_snapshot():
    """
    Pull today s values for all indicators.
    Returns dict: {indicator_name: current_value}
    Used at inference time for real-time stress score.
    """
    snapshot = {}
    snapshot["timestamp"] = datetime.now().isoformat()
    snapshot["market_date"] = TODAY

    live_tickers = {
        "vix_live":    "^VIX",
        "sp500_live":  "^GSPC",
        "gold_live":   "GLD",
        "dollar_live": "DX-Y.NYB",
        "vvix_live":   "^VVIX",
        "bonds_live":  "TLT",
        "hy_bonds":    "HYG",
        "ig_bonds":    "LQD"
    }

    for name, ticker in live_tickers.items():
        try:
            t = yf.Ticker(ticker)
            hist = t.history(period="2d")
            if len(hist) > 0:
                val = hist["Close"].iloc[-1]
                prev = hist["Close"].iloc[-2] if len(hist) > 1 else val
                change_pct = ((val - prev) / prev) * 100
                snapshot[name] = {
                    "value": round(float(val), 2),
                    "change_pct": round(float(change_pct), 2),
                    "date": str(hist.index[-1].date())
                }
        except Exception as e:
            snapshot[name] = {"value": None, "error": str(e)}

    # FRED live values (most recent available)
    fred_live = {
        "yield_curve_live": "T10Y2Y",
        "ted_spread_live":  "TEDRATE",
        "nfci_live":        "NFCI"
    }

    for name, series_id in fred_live.items():
        try:
            series = fred.get_series(
                series_id,
                observation_start=(
                    datetime.today() - timedelta(days=30)
                ).strftime("%Y-%m-%d")
            )
            snapshot[name] = {
                "value": round(float(series.iloc[-1]), 4),
                "date": str(series.index[-1].date())
            }
        except Exception as e:
            snapshot[name] = {"value": None, "error": str(e)}

    return snapshot


live_snapshot = get_live_snapshot()

# Print live snapshot as formatted table
print("=" * 60)
print(f"LIVE MARKET SNAPSHOT -- {TODAY}")
print("=" * 60)

stress_thresholds = {
    "vix_live": 30, "vvix_live": 120, "dollar_live": 105
}

for name, info in live_snapshot.items():
    if name in ("timestamp", "market_date"):
        continue
    if isinstance(info, dict) and info.get("value") is not None:
        val = info["value"]
        chg = info.get("change_pct", 0)
        thresh = stress_thresholds.get(name)
        status = "STRESS" if thresh and val > thresh else "NORMAL"
        print(f"  {name:30s} {val:10.2f}  {chg:+.2f}%  {status}")
    elif isinstance(info, dict):
        print(f"  {name:30s}  -- unavailable --")

---
## Step 5 -- Live News Sentiment

We pull today s financial news headlines via NewsAPI and run FinBERT
(a BERT model fine-tuned on financial text) to classify each headline
as positive, negative, or neutral. The aggregate negative sentiment
becomes our news stress signal (0-100).

### 5A -- Fetch Headlines
Get a free key at: https://newsapi.org (instant, free, 100 requests/day)

In [ ]:
NEWS_API_KEY = "paste_your_key_here"  # Replace with your NewsAPI key

def fetch_financial_news(api_key, days_back=1):
    """
    Fetch financial news headlines from last N days.
    Returns list of headline strings.
    """
    from_date = (
        datetime.today() - timedelta(days=days_back)
    ).strftime("%Y-%m-%d")

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": (
            "stock market OR financial crisis OR "
            "recession OR Federal Reserve OR "
            "inflation OR interest rates OR "
            "market crash OR economic"
        ),
        "from": from_date,
        "sortBy": "publishedAt",
        "language": "en",
        "pageSize": 100,
        "apiKey": api_key
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        articles = response.json().get("articles", [])
        headlines = [
            a["title"] + ". " + (a["description"] or "")
            for a in articles
            if a["title"] and "[Removed]" not in a["title"]
        ]
        print(f"Fetched {len(headlines)} headlines")
        return headlines
    else:
        print(f"NewsAPI error: {response.status_code}")
        return []


headlines = fetch_financial_news(NEWS_API_KEY, days_back=1)

# Show sample headlines
print("\nSample headlines:")
for i, h in enumerate(headlines[:10], 1):
    print(f"  {i}. {h[:120]}")


### 5B -- FinBERT Sentiment Analysis

FinBERT (ProsusAI/finbert) is a BERT model fine-tuned on financial
communication text. It classifies text into positive, negative, or
neutral with calibrated confidence scores.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn.functional as F

# Load FinBERT -- pretrained financial sentiment model
finbert_tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
finbert_model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert")
finbert_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
finbert_model = finbert_model.to(device)
print(f"FinBERT loaded on {device}")


def analyze_sentiment_batch(texts, batch_size=32):
    """
    Run FinBERT on list of headlines.
    Returns list of dicts: {text, positive, negative, neutral, label}
    """
    results = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = finbert_tokenizer(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = finbert_model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)

        for text, prob in zip(batch, probs.cpu()):
            # FinBERT labels: 0=positive, 1=negative, 2=neutral
            results.append({
                "text": text[:100],
                "positive": round(float(prob[0]), 4),
                "negative": round(float(prob[1]), 4),
                "neutral":  round(float(prob[2]), 4),
                "label": ["positive", "negative", "neutral"][prob.argmax().item()]
            })

    return results


print("Running FinBERT on headlines...")
sentiment_results = analyze_sentiment_batch(headlines)
print(f"Analyzed {len(sentiment_results)} headlines")

### 5C -- Compute News Stress Score

We convert the per-headline sentiment scores into a single 0-100 news
stress score. The score reflects the balance of negative vs positive
sentiment across all headlines. A score of 50 indicates neutral; higher
values mean the news flow is more negative/fearful.

In [ ]:
def compute_news_stress_score(sentiment_results):
    """
    Convert FinBERT results to a single stress score (0-100).
    0   = all positive news
    50  = neutral mix
    100 = all negative/crisis news
    """
    if not sentiment_results:
        return 50, [], [], {}

    df_sent = pd.DataFrame(sentiment_results)
    stress_scores = df_sent["negative"] - df_sent["positive"]

    # Normalize to 0-100
    news_stress = float(((stress_scores.mean() + 1) / 2) * 100)
    news_stress = max(0, min(100, news_stress))

    df_sent["stress"] = stress_scores
    top_negative = df_sent.nlargest(5, "stress")[
        ["text", "negative", "stress"]
    ].to_dict("records")
    top_positive = df_sent.nsmallest(5, "stress")[
        ["text", "positive", "stress"]
    ].to_dict("records")
    label_counts = df_sent["label"].value_counts().to_dict()

    return news_stress, top_negative, top_positive, label_counts


news_stress, top_neg, top_pos, label_counts = compute_news_stress_score(
    sentiment_results
)

# Print formatted output
print("=" * 60)
print(f"NEWS SENTIMENT ANALYSIS -- {TODAY}")
print("=" * 60)
print(f"News Stress Score: {news_stress:.1f}/100")
n_pos = label_counts.get("positive", 0)
n_neg = label_counts.get("negative", 0)
n_neu = label_counts.get("neutral", 0)
print(f"Positive headlines: {n_pos}")
print(f"Negative headlines: {n_neg}")
print(f"Neutral headlines:  {n_neu}")
print()
print("TOP STRESS HEADLINES:")
for h in top_neg:
    neg_pct = h["negative"]
    txt = h["text"]
    print(f"  [{neg_pct:.0%} negative] {txt}")
print()
print("TOP POSITIVE HEADLINES:")
for h in top_pos:
    pos_pct = h["positive"]
    txt = h["text"]
    print(f"  [{pos_pct:.0%} positive] {txt}")


---
## Step 6 -- Align and Clean All Historical Data

Different indicators publish at different frequencies (daily, weekly,
monthly). We align all 15 indicators to a common business-day index,
forward-fill gaps up to 5 days (standard financial practice for weekly
releases), and drop rows where more than 30% of indicators are missing.

In [ ]:
def build_master_dataframe(raw_data, start, end):
    """
    Align all 15 indicators to same business-day date index.
    Forward fill missing values (max 5 days) since FRED releases
    data weekly/monthly. Drop dates where >30% of indicators missing.
    """
    bdays = pd.bdate_range(start=start, end=end)

    aligned = {}
    for key, series in raw_data.items():
        if len(series) == 0:
            continue
        series.index = pd.to_datetime(series.index)
        series = series[~series.index.duplicated()]
        # Remove timezone info for alignment
        if series.index.tz is not None:
            series.index = series.index.tz_localize(None)
        reindexed = series.reindex(bdays)
        filled = reindexed.ffill(limit=5)
        aligned[key] = filled

    df = pd.DataFrame(aligned, index=bdays)

    # Drop rows with too many missing values
    threshold = len(df.columns) * 0.7
    df = df.dropna(thresh=int(threshold))

    print(f"Master DataFrame: {df.shape}")
    print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
    print(f"Columns: {list(df.columns)}")
    print("\nMissing values per column:")
    print(df.isnull().sum())

    return df


df = build_master_dataframe(raw_data, START_DATE, TODAY)


---
## Step 7 -- Compute Derived Features

Raw indicator values alone are not enough. A VIX of 28 is less
informative than knowing VIX jumped 40% in 5 days, or that it is
2 standard deviations above its 1-year average. We engineer rolling
z-scores, rates of change, drawdowns, regime indicators, and
velocity features to give the model richer context.

In [ ]:
def add_derived_features(df):
    """
    Add features the model needs beyond raw values.
    Rolling z-scores show how extreme values are vs recent history.
    Rate of change captures momentum and acceleration.
    """
    # 1. Rolling z-scores (how extreme vs 1-year history)
    for col in df.columns:
        roll_mean = df[col].rolling(252).mean()
        roll_std  = df[col].rolling(252).std()
        df[f"{col}_zscore"] = (df[col] - roll_mean) / (roll_std + 1e-8)

    # 2. Rate of change (5-day momentum)
    for col in ["vix", "credit_spread_hy", "yield_curve", "sp500"]:
        if col in df.columns:
            df[f"{col}_roc5"] = df[col].pct_change(5)

    # 3. S&P 500 drawdown from 52-week high
    if "sp500" in df.columns:
        rolling_max = df["sp500"].rolling(252).max()
        df["sp500_drawdown"] = (df["sp500"] - rolling_max) / rolling_max

    # 4. VIX regime flags
    if "vix" in df.columns:
        df["vix_regime"] = (df["vix"] > 30).astype(int)
        df["vix_extreme"] = (df["vix"] > 40).astype(int)

    # 5. Yield curve inversion duration (consecutive days inverted)
    if "yield_curve" in df.columns:
        inverted = (df["yield_curve"] < 0).astype(int)
        df["inversion_duration"] = (
            inverted.groupby((inverted != inverted.shift()).cumsum()).cumcount() + 1
        ) * inverted

    # 6. Credit spread widening speed (21-day change)
    if "credit_spread_hy" in df.columns:
        df["credit_spread_velocity"] = df["credit_spread_hy"].diff(21)

    print(f"Features after engineering: {df.shape[1]}")
    return df


df = add_derived_features(df)

---
## Step 8 -- Label Crisis Periods

Every trading day gets a label: 0 = normal, 1 = stress/pre-crisis,
2 = crisis/acute stress. We also add pre-crisis labels (30 days before
each crisis start) so the model can learn early warning signals.
These labels become the training target for Day 2.

In [ ]:
crisis_periods = [
    # (start, end, name, severity)
    # severity: 1=stress, 2=crisis
    ("1997-07-01", "1997-12-31", "Asian Financial Crisis", 1),
    ("1998-08-01", "1998-10-15", "Russian Crisis / LTCM", 2),
    ("2000-03-10", "2002-10-09", "Dot-com Crash", 2),
    ("2001-09-11", "2001-09-21", "9/11 Market Closure", 2),
    ("2007-07-01", "2007-12-31", "Subprime Mortgage Stress", 1),
    ("2008-01-01", "2009-03-09", "Global Financial Crisis", 2),
    ("2010-04-23", "2010-07-01", "European Debt Crisis", 1),
    ("2011-07-22", "2011-10-03", "US Debt Ceiling Crisis", 1),
    ("2015-08-17", "2015-09-30", "China Devaluation Shock", 1),
    ("2018-10-01", "2018-12-24", "Rate Hike / Trade War", 1),
    ("2020-02-19", "2020-03-23", "COVID-19 Crash", 2),
    ("2020-03-23", "2020-06-01", "COVID Recovery Stress", 1),
    ("2022-01-01", "2022-10-12", "Inflation / Rate Hike Bear", 1),
    ("2023-03-08", "2023-03-31", "SVB Banking Crisis", 2),
]


def label_crisis_periods(df, crisis_periods):
    """
    Label every trading day with crisis severity.
    Also label 30 days before crisis as pre-crisis
    so the model can learn early warning signals.
    """
    df["crisis_label"] = 0
    df["crisis_name"] = "normal"
    df["pre_crisis"] = 0

    for start, end, name, severity in crisis_periods:
        mask = (df.index >= start) & (df.index <= end)
        df.loc[mask, "crisis_label"] = severity
        df.loc[mask, "crisis_name"] = name

        # Label 30 days before crisis as pre-crisis
        pre_start = (
            pd.Timestamp(start) - timedelta(days=30)
        ).strftime("%Y-%m-%d")
        pre_mask = (df.index >= pre_start) & (df.index < start)
        df.loc[pre_mask, "pre_crisis"] = 1
        tag = name.lower().replace(" ", "_")
        df.loc[pre_mask, "crisis_name"] = f"pre_{tag}"

    print("CRISIS LABEL DISTRIBUTION:")
    print(df["crisis_label"].value_counts())
    pre_count = df["pre_crisis"].sum()
    print(f"\nPre-crisis days: {pre_count}")

    return df


df = label_crisis_periods(df, crisis_periods)


---
## Step 9 -- Compute Simple Stress Score

Before the model is trained, we need a simple rule-based stress score
to validate the data and create a baseline. Each indicator is converted
to its historical percentile rank, flipped if low = stress, then
weighted to produce a composite score from 0 to 100.

In [ ]:
def compute_rule_based_stress(df, config):
    """
    Simple weighted percentile-rank stress score.
    Weights: VIX=20%, credit_spread=20%, yield_curve=15%,
    financial_conditions=15%, ted_spread=10%, others=5% each.
    """
    weights = {
        "vix": 0.20, "credit_spread_hy": 0.20,
        "yield_curve": 0.15, "financial_conditions": 0.15,
        "ted_spread": 0.10, "vvix": 0.05,
        "dollar_index": 0.05, "consumer_sentiment": 0.05,
        "unemployment_claims": 0.05
    }

    stress_score = pd.Series(0.0, index=df.index)
    total_weight = 0

    for indicator, weight in weights.items():
        if indicator not in df.columns:
            continue
        series = df[indicator].dropna()
        if len(series) < 252:
            continue

        pct_rank = series.rank(pct=True) * 100
        pct_rank = pct_rank.reindex(df.index)

        cfg = config.get(indicator, {})
        if cfg.get("stress_direction") == "low":
            pct_rank = 100 - pct_rank

        stress_score += pct_rank.fillna(50) * weight
        total_weight += weight

    stress_score = stress_score / total_weight
    df["stress_score"] = stress_score
    return df


df = compute_rule_based_stress(df, indicators_config)
ss_min = df["stress_score"].min()
ss_max = df["stress_score"].max()
ss_cur = df["stress_score"].iloc[-1]
print(f"Stress score range: {ss_min:.1f} -- {ss_max:.1f}")
print(f"Current stress: {ss_cur:.1f}/100")


---
## Step 10 -- Compute Today s Live Stress Score

Combines three components into a single live stress reading:
- Historical indicators (70%): rule-based score from the full dataset
- Live market data (20%): real-time Yahoo Finance values
- News sentiment (10%): FinBERT analysis of today s headlines

In [ ]:
def compute_live_stress(df, live_snapshot, news_stress_score):
    """
    Combine historical indicators (70%), live market data (20%),
    and news sentiment (10%) into a single live stress score.
    """
    historical_stress = float(df["stress_score"].iloc[-1])

    live_signals = []

    vix_live = live_snapshot.get("vix_live", {}).get("value")
    if vix_live:
        vix_historical = df["vix"].dropna()
        vix_pct = float((vix_historical < vix_live).mean() * 100)
        live_signals.append(vix_pct)

    sp500_change = live_snapshot.get("sp500_live", {}).get("change_pct", 0)
    if sp500_change:
        sp500_stress = max(0, min(100, 50 - sp500_change * 5))
        live_signals.append(sp500_stress)

    live_market_stress = np.mean(live_signals) if live_signals else 50

    combined_stress = (
        historical_stress * 0.70 +
        live_market_stress * 0.20 +
        news_stress_score * 0.10
    )
    combined_stress = max(0, min(100, combined_stress))

    return {
        "combined_stress": round(combined_stress, 1),
        "historical_component": round(historical_stress, 1),
        "live_market_component": round(live_market_stress, 1),
        "news_component": round(news_stress_score, 1),
        "timestamp": datetime.now().isoformat()
    }


live_stress = compute_live_stress(df, live_snapshot, news_stress)

score = live_stress["combined_stress"]
hist_c = live_stress["historical_component"]
live_c = live_stress["live_market_component"]
news_c = live_stress["news_component"]

print("=" * 60)
print(f"CRASHSIGNAL LIVE -- {TODAY}")
print("=" * 60)
print(f"OVERALL STRESS SCORE: {score:.1f}/100")
print()

if score <= 25:
    level = "LOW -- Markets calm"
elif score <= 50:
    level = "MODERATE -- Watch closely"
elif score <= 75:
    level = "ELEVATED -- Stress building"
elif score <= 90:
    level = "HIGH -- Crisis conditions"
else:
    level = "EXTREME -- Acute crisis"

print(f"Level: {level}")
print()
print(f"Historical indicators: {hist_c:.1f}")
print(f"Live market data:      {live_c:.1f}")
print(f"News sentiment:        {news_c:.1f}")


---
## Step 11 -- Visualizations

Five visualizations to validate the data pipeline and showcase
the system. All use dark backgrounds and Plotly for interactivity.

### Plot 1: Historical Stress Score vs Crisis Periods

In [ ]:
# Plot 1: Historical Stress Score with Crisis Shading
fig = go.Figure()

# Add stress score line
fig.add_trace(go.Scatter(
    x=df.index, y=df["stress_score"],
    mode="lines", name="Stress Score",
    line=dict(color="#00d4ff", width=1)
))

# Shade crisis periods
colors = {1: "rgba(255, 165, 0, 0.2)", 2: "rgba(255, 0, 0, 0.3)"}
for start, end, name, severity in crisis_periods:
    fig.add_vrect(
        x0=start, x1=end,
        fillcolor=colors[severity],
        layer="below", line_width=0,
        annotation_text=name if severity == 2 else "",
        annotation_position="top left",
        annotation_font_size=8,
        annotation_font_color="white"
    )

# Threshold lines
for val, label, color in [(50, "Moderate", "yellow"), (75, "High", "orange"), (90, "Extreme", "red")]:
    fig.add_hline(y=val, line_dash="dash", line_color=color,
                  annotation_text=label, annotation_font_color=color)

fig.update_layout(
    title="CrashSignal: 30 Years of Market Stress",
    xaxis_title="Date", yaxis_title="Stress Score (0-100)",
    template="plotly_dark", height=500,
    yaxis=dict(range=[0, 100])
)
fig.show()

### Plot 2: Indicator Heatmap by Year

In [ ]:
# Plot 2: Indicators Heatmap (annual averages, normalized)
raw_cols = [c for c in indicators_config.keys() if c in df.columns]
df_annual = df[raw_cols].resample("YE").mean()

# Normalize each column to 0-1 for heatmap
df_norm = df_annual.apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))

# Flip low-stress-direction indicators
for col in raw_cols:
    cfg = indicators_config.get(col, {})
    if cfg.get("stress_direction") == "low":
        df_norm[col] = 1 - df_norm[col]

df_norm.index = df_norm.index.year

fig2 = go.Figure(data=go.Heatmap(
    z=df_norm.T.values,
    x=df_norm.index.astype(str),
    y=[indicators_config[c]["name"][:30] for c in raw_cols],
    colorscale="YlOrRd",
    colorbar_title="Stress Level"
))

fig2.update_layout(
    title="Indicator Stress Levels by Year",
    template="plotly_dark", height=500
)
fig2.show()

### Plot 3: Correlation Matrix (Normal vs Crisis)

In [ ]:
# Plot 3: Correlation matrices -- normal vs crisis
raw_cols = [c for c in indicators_config.keys() if c in df.columns]
short_names = [indicators_config[c]["name"][:20] for c in raw_cols]

corr_normal = df.loc[df["crisis_label"] == 0, raw_cols].corr()
corr_crisis = df.loc[df["crisis_label"] > 0, raw_cols].corr()

fig3 = make_subplots(rows=1, cols=2,
    subplot_titles=["Normal Periods", "Crisis Periods"])

fig3.add_trace(go.Heatmap(
    z=corr_normal.values, x=short_names, y=short_names,
    colorscale="RdBu_r", zmin=-1, zmax=1, showscale=False
), row=1, col=1)

fig3.add_trace(go.Heatmap(
    z=corr_crisis.values, x=short_names, y=short_names,
    colorscale="RdBu_r", zmin=-1, zmax=1
), row=1, col=2)

fig3.update_layout(
    title="Indicator Correlations: Normal vs Crisis",
    template="plotly_dark", height=500
)
fig3.show()

### Plot 4: Today s Indicator Dashboard

In [ ]:
# Plot 4: Current values vs historical average
raw_cols = [c for c in indicators_config.keys() if c in df.columns]

current_vals = []
hist_means   = []
bar_colors   = []
names        = []

for col in raw_cols:
    cfg = indicators_config[col]
    cur = df[col].dropna().iloc[-1] if len(df[col].dropna()) > 0 else 0
    avg = df[col].mean()
    thresh = cfg.get("stress_threshold")

    stressed = False
    if thresh is not None:
        if cfg["stress_direction"] == "high" and cur > thresh:
            stressed = True
        elif cfg["stress_direction"] == "low" and cur < thresh:
            stressed = True

    # Normalize for display
    col_max = df[col].max()
    col_min = df[col].min()
    rng = col_max - col_min if col_max != col_min else 1
    current_vals.append((cur - col_min) / rng * 100)
    hist_means.append((avg - col_min) / rng * 100)
    bar_colors.append("#ff4444" if stressed else "#44ff88")
    names.append(cfg["name"][:25])

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    y=names, x=current_vals, orientation="h",
    name="Current", marker_color=bar_colors
))
fig4.add_trace(go.Scatter(
    y=names, x=hist_means, mode="markers",
    name="Historical Avg", marker=dict(color="white", size=10, symbol="diamond")
))

fig4.update_layout(
    title=f"Today s Indicators vs Historical Average ({TODAY})",
    xaxis_title="Normalized Level (0-100)",
    template="plotly_dark", height=600,
    legend=dict(orientation="h", y=1.1)
)
fig4.show()

### Plot 5: Crisis Timeline

In [ ]:
# Plot 5: Crisis timeline (horizontal bars)
fig5 = go.Figure()

severity_colors = {1: "#FFA500", 2: "#FF2222"}
severity_labels = {1: "Stress", 2: "Crisis"}

for i, (start, end, name, sev) in enumerate(crisis_periods):
    fig5.add_trace(go.Bar(
        y=[name],
        x=[(pd.Timestamp(end) - pd.Timestamp(start)).days],
        base=[pd.Timestamp(start).toordinal()],
        orientation="h",
        marker_color=severity_colors[sev],
        name=severity_labels[sev],
        showlegend=(i < 2),
        hovertemplate=f"{name}<br>{start} to {end}<br>{severity_labels[sev]}<extra></extra>"
    ))

# Convert ordinal x-axis to date labels
tick_years = list(range(1994, 2026, 2))
tick_vals = [pd.Timestamp(f"{y}-01-01").toordinal() for y in tick_years]
tick_text = [str(y) for y in tick_years]

fig5.update_layout(
    title="Crisis Timeline: 1994-2024",
    xaxis=dict(tickvals=tick_vals, ticktext=tick_text, title="Year"),
    template="plotly_dark", height=500,
    barmode="overlay"
)
fig5.show()

---
## Step 12 -- Save All Artifacts

Save everything to /kaggle/working/ for Day 2 training:
1. **master_df.parquet** -- Full cleaned DataFrame (all indicators + features + labels)
2. **live_snapshot.json** -- Today s live market values
3. **news_sentiment.json** -- Today s headline sentiments and stress score
4. **live_stress.json** -- Today s final combined stress score
5. **crisis_periods.json** -- All labeled crisis periods for visualization
6. **feature_columns.json** -- List of all feature column names needed by model

In [ ]:
import json

out = OUTPUT_DIR

# 1. Master DataFrame
df.to_parquet(os.path.join(out, "master_df.parquet"))
print("[SAVED] master_df.parquet")

# 2. Live snapshot
with open(os.path.join(out, "live_snapshot.json"), "w") as f:
    json.dump(live_snapshot, f, indent=2, default=str)
print("[SAVED] live_snapshot.json")

# 3. News sentiment
with open(os.path.join(out, "news_sentiment.json"), "w") as f:
    json.dump({
        "headlines": sentiment_results[:50],
        "stress_score": news_stress,
        "label_counts": label_counts,
        "top_negative": top_neg,
        "top_positive": top_pos,
        "timestamp": TODAY
    }, f, indent=2)
print("[SAVED] news_sentiment.json")

# 4. Live stress
with open(os.path.join(out, "live_stress.json"), "w") as f:
    json.dump(live_stress, f, indent=2)
print("[SAVED] live_stress.json")

# 5. Crisis periods
with open(os.path.join(out, "crisis_periods.json"), "w") as f:
    json.dump([
        {"start": s, "end": e, "name": n, "severity": sv}
        for s, e, n, sv in crisis_periods
    ], f, indent=2)
print("[SAVED] crisis_periods.json")

# 6. Feature columns
feature_cols = [
    c for c in df.columns
    if c not in ["crisis_label", "crisis_name", "pre_crisis", "stress_score"]
]
with open(os.path.join(out, "feature_columns.json"), "w") as f:
    json.dump(feature_cols, f, indent=2)
print("[SAVED] feature_columns.json")

# Final summary
n_days = len(df)
n_indicators = sum(1 for v in raw_data.values() if len(v) > 0)
n_crises = len(crisis_periods)
n_features = len(feature_cols)
combined = live_stress["combined_stress"]

print()
print("=" * 60)
print("DAY 1 COMPLETE")
print("=" * 60)
print(f"Historical data: {n_days} trading days")
print(f"Indicators: {n_indicators} pulled successfully")
print(f"Crisis periods labeled: {n_crises}")
print(f"Features engineered: {n_features}")
print()
print(f"TODAY S LIVE STRESS: {combined}/100")
print(f"TODAY S NEWS STRESS: {news_stress:.1f}/100")
print()
print("Files saved:")
print(f"  master_df.parquet     ({df.shape})")
print("  live_snapshot.json")
print("  news_sentiment.json")
print("  live_stress.json")
print("  crisis_periods.json")
print("  feature_columns.json")
print()
print("NEXT: Day 2 -- Train Temporal Fusion Transformer")
print("=" * 60)
